## HALI — Week 6 MCP (community contribution)

**HALI** = HPV Awareness & Learning Initiative for Community Health Volunteers in Kenya.

This notebook runs an **OpenAI Agents SDK** agent that talks to a **local MCP server** (`hali_mcp_server.py`) over **stdio**.

**Run with working directory** = `6_mcp/community_contributions/stellaoiro` (same folder as the server file).

In [ ]:
# 1. Imports
from pathlib import Path
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

if not Path("hali_mcp_server.py").exists():
    raise FileNotFoundError("Start Jupyter from the folder that contains hali_mcp_server.py")

In [ ]:
# 2. MCP server params — spawns hali_mcp_server.py as a subprocess over stdio
params = {"command": "uv", "args": ["run", "hali_mcp_server.py"]}

In [ ]:
# 3. Inspect tools exposed by the MCP server
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    tools = await server.list_tools()

tools

In [ ]:
# 4. Agent instructions and reusable helper
instructions = """You are HALI, supporting Community Health Volunteers in Kenya with HPV vaccine outreach.

Key facts you know and can state confidently:
- The HPV vaccine does NOT cause infertility — this is a myth with no scientific basis
- It prevents cervical cancer, which kills 3,400 Kenyan women every year
- One dose is enough for girls aged 9-14; it is free at public health facilities
- Islamic and Christian leaders in Kenya support vaccination
- Side effects are mild (sore arm, brief fever) and temporary

Use the tools:
- check_eligibility — when you have age (and optionally gender)
- record_interest — when someone wants vaccination or follow-up
- record_unknown_question — only for questions outside your knowledge; never invent medical facts

Reply warmly and briefly; prefer Kiswahili or English as appropriate."""

model = "gpt-4o-mini"

async def ask_hali(request: str) -> str:
    async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
        agent = Agent(name="HALI", instructions=instructions, model=model, mcp_servers=[mcp_server])
        with trace("hali_mcp"):
            result = await Runner.run(agent, request)
    return result.final_output

In [ ]:
# 5. Test — eligibility check (calls check_eligibility tool)
display(Markdown(await ask_hali(
    "A mother asks: My daughter is 12. Is she eligible for the free HPV vaccine in Kenya?"
)))

In [ ]:
# 6. Test — myth busting (answered from instructions, no tool needed)
display(Markdown(await ask_hali(
    "Someone asks: Will the vaccine make her infertile?"
)))

In [ ]:
# 7. Test — record interest (calls record_interest tool)
display(Markdown(await ask_hali(
    "Grace Otieno in Kisumu wants her 13-year-old daughter vaccinated. Her number is 0712345678. Please log her for follow-up."
)))